# OpenPlaque V22 — full-volume frozen V21 inference on DEV80

Research-only **no-training** experiment. Uses the exact completed V21 six-epoch checkpoint, the frozen 480/80 split, and the original NIfTI CT at native resolution. No label-guided crops, manual coordinates or sliders.

**Why:** V21's positive-centered patch metrics do not establish whole-volume behavior. V22 measures annotated-positive recall, overlap with other known arteries, raw prediction volume, multi-head conflicts, low-HU artifacts, and 6-connected predicted components. Unannotated tissue is **unknown, not automatically false positive**. No verified negative anatomy or expert review is claimed.

**Instructions:** choose a Colab **T4 or L4 GPU** and select **Runtime → Run all**. Mount Drive is the first executable cell, followed immediately by reuse settings. Smoke test and all 80 cases are resumable. Source masks and the sealed test set remain untouched.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Reuse and run configuration. Preserve all frozen dataset/model settings.
RUN_SMOKE_ONE_CASE = True
RUN_COMPLETE_DEV80 = True
REUSE_VALID_CACHES = True
BATCH_SIZE = 4  # T4/L4. Reduce to 2 only if GPU memory is insufficient.
print("V22 full FOV inference only; no training or test-set reads.")

In [ ]:
import sys, subprocess, importlib.util, torch
missing=[m for m in ('nibabel','scipy') if importlib.util.find_spec(m) is None]
if missing: subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print('PyTorch',torch.__version__,'CUDA available?',torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Please select a GPU runtime (T4/L4); full-volume inference requires CUDA.')

In [ ]:
from pathlib import Path
import hashlib, json, subprocess, sys
ROOT=Path('/content/drive/MyDrive/OpenPlaque')
SCRIPT=ROOT/'OpenPlaque_Evaluation_Dataset_v0_2/openplaque_v22_eval_20261012.py'
EXPECTED_SHA='5db68089418cc3565c6691535a152ea7d642dcc37f83aede74505c6a922dfa10'
if not SCRIPT.is_file():raise FileNotFoundError(f'Pinned V22 evaluator missing: {SCRIPT}')
actual=hashlib.sha256(SCRIPT.read_bytes()).hexdigest()
if actual!=EXPECTED_SHA:raise RuntimeError(f'STOP V22 script checksum mismatch: {actual}')
def run(*args):
    command=[sys.executable,str(SCRIPT),*list(map(str,args))]
    print('V22',command,flush=True)
    subprocess.run(command,check=True)
print('Pinned V22 evaluator verified:',EXPECTED_SHA)

In [ ]:
run('test')
run('preflight')
print('All preflight gates passed: frozen V21, DEV80, original V7 screening, no test access.')

In [ ]:
if RUN_SMOKE_ONE_CASE:
    run('all','--max-cases','1','--batch-size',BATCH_SIZE)
    print('Full-native-volume smoke finished. The case will be reused by the complete run.')

In [ ]:
if RUN_COMPLETE_DEV80:
    args=['all','--batch-size',str(BATCH_SIZE)]
    if not REUSE_VALID_CACHES:args.append('--force')
    run(*args)
else:
    print('Full cohort disabled; only smoke results present.')

In [ ]:
run('report')
output=ROOT/'OpenPlaque_Evaluation_Dataset_v0_2/evaluation_v22_full_volume'
summary=json.loads((output/'whole_volume_summary_v22.json').read_text())
print('Cases:',summary['cases_complete'],'/',summary['cases_expected'],' complete:',summary['complete'])
print(json.dumps(summary['per_artery_reference_conditioned_metrics'],indent=2))
from IPython.display import HTML,display
report=output/'OPENPLAQUE_V22_WHOLE_VOLUME_REPORT.html'
if report.exists():display(HTML(report.read_text()))
print('Saved to',output)
print('Research-only metrics. No negative ground truth or clinical artery identity claims.')